<a href="https://colab.research.google.com/github/manoj-1229/open-data-intelligence-hub/blob/main/task_5_ecommerce_recommendation_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Set reproducibility seed and size
np.random.seed(42)
data_size = 1000

# 2. Generate features
user_ids = [f"USR_{i:04d}" for i in range(1, data_size + 1)]
product_ids = [f"PROD_{np.random.randint(100, 500):03d}" for _ in range(data_size)]
categories = np.random.choice(['Electronics', 'Clothing', 'Home', 'Beauty', 'Books'], size=data_size)
genders = np.random.choice(['Male', 'Female', 'Other'], size=data_size, p=[0.48, 0.48, 0.04])
locations = np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Miami'], size=data_size)

age = np.random.choice([18, 24, 35, 45, 50, 65, np.nan], size=data_size, p=[0.15, 0.2, 0.25, 0.15, 0.1, 0.1, 0.05])
price = np.round(np.random.uniform(10.0, 500.0, size=data_size), 2)
browsing_time = np.random.choice([5.2, 12.5, 25.0, 45.8, 60.2, np.nan], size=data_size, p=[0.2, 0.3, 0.25, 0.15, 0.05, 0.05])
previous_purchases = np.random.randint(0, 20, size=data_size)
discount_applied = np.random.choice([1, 0], size=data_size, p=[0.3, 0.7])
total_spending = np.round((previous_purchases * np.random.uniform(15, 40)) + np.random.uniform(10, 50), 2)
cart_addition = np.random.choice([1, 0], size=data_size, p=[0.4, 0.6])

# 3. Create structural mathematical links for targets
purchase_prob = 0.1 + (cart_addition * 0.4) + (np.nan_to_num(browsing_time) / 150) + (discount_applied * 0.15)
purchase_prob = np.clip(purchase_prob, 0, 1)
purchase_status = np.random.binomial(1, purchase_prob)

rating_base = 3.5 + (discount_applied * 0.5) - (price / 1000) + np.random.normal(0, 0.3, size=data_size)
rating = np.clip(np.round(rating_base, 1), 1.0, 5.0)

# 4. Assemble DataFrame
df = pd.DataFrame({
    'User_ID': user_ids, 'Product_ID': product_ids, 'Category': categories, 'Price': price,
    'Rating': rating, 'Browsing_Time': browsing_time, 'Previous_Purchases': previous_purchases,
    'Cart_Addition': cart_addition, 'Purchase_Status': purchase_status, 'Age': age,
    'Gender': genders, 'Location': locations, 'Discount_Applied': discount_applied, 'Total_Spending': total_spending
})

# 5. Define tracking column segments explicitly
num_features = ['Price', 'Browsing_Time', 'Previous_Purchases', 'Age', 'Total_Spending', 'Discount_Applied']
cat_features = ['Category', 'Gender', 'Location']

# 6. Build automated preprocessing block
num_pipeline = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipeline = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', num_pipeline, num_features), ('cat', cat_pipeline, cat_features)])

print("✅ Step 1 complete: Dataset generated and Preprocessor set up cleanly!")

✅ Step 1 complete: Dataset generated and Preprocessor set up cleanly!


In [7]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Isolate features and target
X_reg = df[num_features + cat_features]
y_reg = df['Rating']

# Split data
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# Connect to pipeline architecture
reg_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge())
])

# Define GridSearchCV hyperparameter optimization space
param_grid_reg = {'model__alpha': [0.1, 1.0, 10.0, 100.0]}

# Train and tune model
grid_reg = GridSearchCV(reg_pipeline, param_grid_reg, cv=5, scoring='neg_mean_absolute_error')
grid_reg.fit(X_train_reg, y_train_reg)

# Evaluate performance
best_reg_model = grid_reg.best_estimator_
y_pred_reg = best_reg_model.predict(X_test_reg)

mae = mean_absolute_error(y_test_reg, y_pred_reg)
mse = mean_squared_error(y_test_reg, y_pred_reg)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred_reg)

print("🎯 PART A: REGRESSION MODEL RESULTS")
print("-" * 40)
print(f"Best Hyperparameter (alpha):   {grid_reg.best_params_['model__alpha']}")
print(f"Mean Absolute Error (MAE):      {mae:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
print(f"R² Score:                       {r2:.3f}")
print(f"\nExample Prediction: Predicted Rating = {y_pred_reg[0]:.1f} out of 5 (Actual: {y_test_reg.iloc[0]})")

🎯 PART A: REGRESSION MODEL RESULTS
----------------------------------------
Best Hyperparameter (alpha):   100.0
Mean Absolute Error (MAE):      0.276
Root Mean Squared Error (RMSE): 0.340
R² Score:                       0.404

Example Prediction: Predicted Rating = 3.7 out of 5 (Actual: 4.0)


In [8]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

# Isolate features and target
X_clf = df[num_features + cat_features]
y_clf = df['Purchase_Status']

# Split data with stratification to maintain balance
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

# Setup model pipeline
clf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

# Define optimization grid
param_grid_clf = {
    'model__C': [0.01, 0.1, 1.0, 10.0],
    'model__solver': ['lbfgs', 'liblinear']
}

# Run optimization search
grid_clf = GridSearchCV(clf_pipeline, param_grid_clf, cv=5, scoring='f1')
grid_clf.fit(X_train_clf, y_train_clf)

# Extract optimized estimations
best_clf_model = grid_clf.best_estimator_
y_pred_clf = best_clf_model.predict(X_test_clf)
y_prob_clf = best_clf_model.predict_proba(X_test_clf)[:, 1]

# Display metrics
print("🎯 PART B: CLASSIFICATION MODEL RESULTS")
print("-" * 40)
print(f"Best Hyperparameters:           {grid_clf.best_params_}")
print(f"Overall Accuracy:               {accuracy_score(y_test_clf, y_pred_clf):.3f}")
print(f"ROC-AUC Score:                  {roc_auc_score(y_test_clf, y_prob_clf):.3f}\n")
print("Detailed Classification Report:")
print(classification_report(y_test_clf, y_pred_clf))

🎯 PART B: CLASSIFICATION MODEL RESULTS
----------------------------------------
Best Hyperparameters:           {'model__C': 10.0, 'model__solver': 'liblinear'}
Overall Accuracy:               0.590
ROC-AUC Score:                  0.529

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.77      0.68       111
           1       0.56      0.36      0.44        89

    accuracy                           0.59       200
   macro avg       0.58      0.57      0.56       200
weighted avg       0.58      0.59      0.57       200



In [9]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Isolate the behavior metrics requested by the business case guidelines
# We extract data processed by our numerical pipeline to ensure balanced K-Means distance metrics
X_preprocessed = preprocessor.fit_transform(df)

# Extract only the numerical column representations for behavioral features
# (Columns 0 to 4 correspond to Price, Browsing_Time, Previous_Purchases, Age, Total_Spending)
X_cluster = X_preprocessed[:, :5]

# 2. Fit K-Means with the business target number of clusters (K = 4 segments)
kmeans = KMeans(n_clusters=4, init='k-means++', max_iter=300, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster)

# 3. Add labels back to the original dataframe for business interpretation
df['Cluster'] = cluster_labels

# 4. Calculate Clustering Evaluation Metrics
inertia = kmeans.inertia_
sil_score = silhouette_score(X_cluster, cluster_labels, sample_size=500, random_state=42)

print("🎯 PART C: CUSTOMER SEGMENTATION RESULTS")
print("-" * 40)
print(f"Number of Customer Segments: {kmeans.n_clusters}")
print(f"Cluster Inertia (Within-Cluster Sum of Squares): {inertia:.2f}")
print(f"Silhouette Score (Cluster Separation):          {sil_score:.3f}\n")

print("📋 Business Interpretation of Customer Segments:")
print("-" * 40)
cluster_summary = df.groupby('Cluster')[['Browsing_Time', 'Total_Spending', 'Previous_Purchases', 'Discount_Applied']].mean()
for cluster_num in range(4):
    print(f"\n🔹 Segment {cluster_num}:")
    print(f"   - Average Browsing Time: {cluster_summary.loc[cluster_num, 'Browsing_Time']:.1f} mins")
    print(f"   - Average Total Spending: ${cluster_summary.loc[cluster_num, 'Total_Spending']:.2f}")
    print(f"   - Avg Past Purchases:    {cluster_summary.loc[cluster_num, 'Previous_Purchases']:.1f}")
    print(f"   - Discount Response Rate: {cluster_summary.loc[cluster_num, 'Discount_Applied']*100:.1f}%")

🎯 PART C: CUSTOMER SEGMENTATION RESULTS
----------------------------------------
Number of Customer Segments: 4
Cluster Inertia (Within-Cluster Sum of Squares): 2629.71
Silhouette Score (Cluster Separation):          0.233

📋 Business Interpretation of Customer Segments:
----------------------------------------

🔹 Segment 0:
   - Average Browsing Time: 18.3 mins
   - Average Total Spending: $338.84
   - Avg Past Purchases:    15.5
   - Discount Response Rate: 30.6%

🔹 Segment 1:
   - Average Browsing Time: 50.4 mins
   - Average Total Spending: $176.14
   - Avg Past Purchases:    6.9
   - Discount Response Rate: 32.0%

🔹 Segment 2:
   - Average Browsing Time: 14.2 mins
   - Average Total Spending: $184.00
   - Avg Past Purchases:    7.3
   - Discount Response Rate: 31.0%

🔹 Segment 3:
   - Average Browsing Time: 14.5 mins
   - Average Total Spending: $138.30
   - Avg Past Purchases:    4.9
   - Discount Response Rate: 31.8%


In [10]:
import pandas as pd

# Compile all high-level data points into a clean data frame
summary_data = {
    "ML Task": ["Rating Prediction (Regression)", "Purchase Likelihood (Classification)", "Customer Segmentation (Clustering)"],
    "Algorithm Used": ["Ridge Regression", "Logistic Regression", "K-Means Clustering"],
    "Primary Metrics": [f"MAE: {mae:.3f} | RMSE: {rmse:.3f} | R²: {r2:.3f}", f"Accuracy: {accuracy:.3f} | ROC-AUC: {roc_auc:.3f}", f"Inertia: {inertia:.1f} | Silhouette: {sil_score:.3f}"],
    "Core Business Value": [
        "Recommends targeted items that users are likely to rate highly.",
        "Identifies buyers for targeted promotions & cart recovery campaigns.",
        "Groups shoppers to deliver personalized discounts and loyalty rewards."
    ]
}

comparison_table = pd.DataFrame(summary_data)
pd.set_option('display.max_colwidth', None)
print("📊 FINAL MODEL COMPARISON & BUSINESS ALIGNMENT TABLE")
print("=" * 80)
print(comparison_table)

NameError: name 'accuracy' is not defined

In [11]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# =====================================================================
# 1. PART C: IMPLEMENT K-MEANS CLUSTERING
# =====================================================================
# Isolate behavior columns from our preprocessing matrix
X_preprocessed = preprocessor.transform(df)
X_cluster = X_preprocessed[:, :5] # Extract numerical feature columns

# Fit K-Means with the business target structure (K = 4 segments)
kmeans = KMeans(n_clusters=4, init='k-means++', max_iter=300, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_cluster)
df['Cluster'] = cluster_labels

# Calculate clustering tracking statistics
inertia_val = kmeans.inertia_
sil_score_val = silhouette_score(X_cluster, cluster_labels, sample_size=500, random_state=42)

print("🎯 PART C: CUSTOMER SEGMENTATION RESULTS")
print("-" * 50)
print(f"Number of Customer Segments: {kmeans.n_clusters}")
print(f"Cluster Inertia:            {inertia_val:.2f}")
print(f"Silhouette Score:           {sil_score_val:.3f}\n")


# =====================================================================
# 2. PART E: AUTOMATED MODEL COMPARISON & ALIGNMENT TABLE
# =====================================================================
# Pull variables directly from the estimators to avoid any memory issues
final_mae = mae # from cell 2
final_rmse = rmse # from cell 2
final_r2 = r2 # from cell 2

# Dynamically pull classification parameters from your active grid model
final_accuracy = grid_clf.best_estimator_.score(X_test_clf, y_test_clf)
from sklearn.metrics import roc_auc_score
final_roc_auc = roc_auc_score(y_test_clf, grid_clf.best_estimator_.predict_proba(X_test_clf)[:, 1])

summary_matrix = {
    "ML Task": [
        "Rating Prediction (Regression)",
        "Purchase Likelihood (Classification)",
        "Customer Segmentation (Clustering)"
    ],
    "Algorithm Used": ["Ridge Regression", "Logistic Regression", "K-Means Clustering"],
    "Primary Metrics": [
        f"MAE: {final_mae:.3f} | RMSE: {final_rmse:.3f} | R²: {final_r2:.3f}",
        f"Accuracy: {final_accuracy:.3f} | ROC-AUC: {final_roc_auc:.3f}",
        f"Inertia: {inertia_val:.1f} | Silhouette: {sil_score_val:.3f}"
    ],
    "Core Business Value": [
        "Recommends targeted items that users are likely to rate highly.",
        "Identifies buyers for targeted promotions & cart recovery campaigns.",
        "Groups shoppers to deliver personalized discounts and loyalty rewards."
    ]
}

comparison_table = pd.DataFrame(summary_matrix)
pd.set_option('display.max_colwidth', None)

print("\n📊 FINAL MODEL COMPARISON & BUSINESS ALIGNMENT TABLE")
print("=" * 90)
print(comparison_table)

🎯 PART C: CUSTOMER SEGMENTATION RESULTS
--------------------------------------------------
Number of Customer Segments: 4
Cluster Inertia:            2629.71
Silhouette Score:           0.233


📊 FINAL MODEL COMPARISON & BUSINESS ALIGNMENT TABLE
                                ML Task       Algorithm Used  \
0        Rating Prediction (Regression)     Ridge Regression   
1  Purchase Likelihood (Classification)  Logistic Regression   
2    Customer Segmentation (Clustering)   K-Means Clustering   

                        Primary Metrics  \
0  MAE: 0.276 | RMSE: 0.340 | R²: 0.404   
1      Accuracy: 0.590 | ROC-AUC: 0.529   
2   Inertia: 2629.7 | Silhouette: 0.233   

                                                      Core Business Value  
0         Recommends targeted items that users are likely to rate highly.  
1    Identifies buyers for targeted promotions & cart recovery campaigns.  
2  Groups shoppers to deliver personalized discounts and loyalty rewards.  
